<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/hallucination_filter_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title requirements
!pip install sentence-transformers -q

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
#@title Confirm that the shared results directory exists
#@markdown Verify the directory path on your Google Drive, and change it if needed
import os
shared_path = "/content/drive/MyDrive/ColabNotebooks/SigExt/experiments" #@param{type:"string"}
print(os.path.exists(shared_path))

True


In [ ]:
#@title Check if the required directories are present
base_path = shared_path

# check all needed files exist
files_to_check = [
    f"{base_path}/cnn_dataset_with_keyphrase/test.jsonl",
    f"{base_path}/cnn_extsig_predictions/test_predictions.json",
    f"{base_path}/cnn_extsig_predictions/test_dataset.jsonl",
    f"{base_path}/cnn_extsig_predictions/test_metrics.json",
]

for f in files_to_check:
  exists = os.path.exists(f)
  print(f"{'✓' if exists else '✗'}  {f}")

✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_dataset_with_keyphrase/test.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions/test_predictions.json
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions/test_dataset.jsonl
✓  /content/drive/MyDrive/ColabNotebooks/SigExt/experiments/cnn_extsig_predictions/test_metrics.json


#Clone the repo

In [ ]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"

# Preparing Data & Pipelines

In [ ]:
# @title Download requirements for data preparing

%cd {path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

[Errno 2] No such file or directory: '/content/drive/MyDrive/DNLP-storage/SigExt'
/content


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import os
import json
import nltk #imports the Natural Language Toolkit used to split text into sentences
import torch
import numpy as np
import tqdm #progress bar library
import jsonlines
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util #util provides the cosine similarity
from transformers import pipeline as hf_pipeline #provides AI models

#define all paths
base = shared_path + "/experiments"
path_scored_dataset   = f"{base}/cnn_dataset_with_keyphrase/test.jsonl" #Path to the test articles that have phrase scores added by the Longformer
path_predictions      = f"{base}/cnn_extsig_predictions/test_predictions.json" #Path to the 500 raw summaries generated by Mistral
path_full_dataset     = f"{base}/cnn_extsig_predictions/test_dataset.jsonl" #Path to the full dataset file that includes raw_output
path_baseline_metrics = f"{base}/cnn_extsig_predictions/test_metrics.json" #Path to the ROUGE scores that were computed by the original SigExt pipeline
path_output           = f"{base}/hallucination_extension/cnn_verified_predictions/" #Path to the folder where the extension will save its results

os.makedirs(path_output, exist_ok=True)

TOP_N_EVIDENCE     = 3 #Controls how many source sentences are retrieved as evidence for each summary sentence
ENTAIL_THRESHOLD   = 0.5 #The minimum confidence score the NLI model must assign to "entailment" for a sentence to pass
MAX_REGEN_ATTEMPTS = 2 #How many times the extension tries to regenerate a flagged sentence before giving up and dropping it

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cpu


In [ ]:
#@title Load Models
print("Loading sentence embedder...")
embedder = SentenceTransformer("all-MiniLM-L6-v2") #Downloads and loads the sentence embedding model, used to produce meaningful sentence embeddings
embedder = embedder.to(device)

print("Loading NLI model...")
#Instead of manually tokenizing input, running the model, and decoding output, the pipeline handles everything automatically
nli_model = hf_pipeline(
    "text-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768",
    device=0 if device == "cuda" else -1,
    top_k=None,
    batch_size=8,
    truncation=True,
    max_length=512,
)


In [1]:
#@title Load Data

#raw summaries generated by the LLMs
with open(path_predictions) as f:
  raw_summaries = json.load(f)

#source articles with phrase scores
with jsonline.open(path_scored_dataset) as f:
  scored_dataset = list(f)

#full dataset with raw_output for the ROUGE evaluation
with jsonline.open(path_full_dataset) as f:
  full_dataset = list(f)

#baseline metrics for comparison
with open(path_baseline_metrics) as f:
  baseline_metrics = json.load(f)

print(f"Summaries loaded:       {len(raw_summaries)}")
print(f"Scored articles loaded: {len(scored_dataset)}")
print(f"Full dataset loaded:    {len(full_dataset)}")
print(f"\nBaseline ROUGE-1 F1: {baseline_metrics.get('rouge1f', 'N/A')}")
print(f"\nSample raw summary:\n{raw_summaries[0]}")

NameError: name 'path_predictions' is not defined

#Verification Functions

In [ ]:
#sentence segmentation
def split_sentences(text):
  """
    Splits a given block of text into a list of individual sentences.

    Args:
        text (str): The input text block to be split.

    Returns:
        list[str]: A list of individual sentences, with leading and trailing
                   whitespace removed. Empty strings are excluded.

    Example:
        >>> split_sentences("Hello world! How are you? ")
        ['Hello world!', 'How are you?']
    """
  return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]


# evidence retrieval
def retrieve_evidence(summary_sentence, source_sentences, top_n=TOP_N_EVIDENCE):
  """
    Retrieves the most semantically similar evidence from a source article.

    Args:
        summary_sentence (str): One sentence from the generated summary.
        source_sentences (list[str]): The list of sentences from the source article.
        top_n (int, optional): The maximum number of evidence sentences to return.
                               Defaults to TOP_N_EVIDENCE (3).

    Returns:
        list[str]: A list containing the top `top_n` matching source sentences.
    """
  query_emb  = embedder.encode(summary_sentence, convert_to_tensor=True) #we need a tensor for the cos_sim
  source_emb = embedder.encode(source_sentences, convert_to_tensor=True)
  scores     = util.cos_sim(query_emb, source_emb) #cosine similarity between the  the summary sentence vector and every source sentence vector
  top_idx    = torch.topk(scores, k=min(top_n, len(source_sentence))).indices
  return [source_sentences[i] for i in top_idx]



#NLI entailment check
def check_entailment(summary_sentence, evidence):
  """
    Evaluates whether the source evidence supports the generated summary sentence.

    This function uses a Natural Language Inference (NLI) model to compare the
    evidence (premise) against the summary (hypothesis). It classifies the
    relationship as 'entailed', 'contradiction', or 'neutral'.

    Args:
        summary_sentence (str): The generated sentence being fact-checked (the hypothesis).
        evidence (list[str]): The list of source sentences acting as proof (the premise).

    Returns:
        tuple[str, float]: A tuple containing the final classification label
                           ("entailed", "contradiction", or "neutral") and its
                           corresponding confidence score.
    """
  premise = " ".join(evidence)
  result  = nli_model({"text": premise, "text_pair": summary_sentence}) #text is the premise, while text_pair is the hypothesis

  label_scores = {item["label"]: item["score"] for item in result}

  entail_score = label_scores.get("entailment", 0.0)
  if entail_score >= ENTAIL_THRESHOLD:
    return "entailed", entail_score
  elif label_scores.get("contradiction", 0) > label_scores.get("neutral",0):
    return "contradiction", label_scores.get("contradiction",0.0)
  else:
    return "neutral", label_scores.get("neutral",0.0)


#regenerate sentence (to be modified in the future
def regenerate_sentence(original_sentence: str, evidence: list[str]) -> str | None:
    """
    Prompts an LLM to rewrite an unsupported sentence using only verified evidence.

    If a generated sentence fails the NLI entailment check, this function attempts
    to salvage it by forcing a generation model to rewrite it strictly based on
    the provided source evidence.

    Args:
        original_sentence (str): The hallucinated or unsupported summary sentence.
        evidence (list[str]): The source sentences to use as ground-truth context.

    Returns:
        str | None: The rewritten, factually correct sentence. Returns None if
                    the model cannot rewrite it or explicitly outputs "UNSUPPORTED".
    """
    evidence_text = " ".join(evidence)
    prompt = (
        f"The following sentence may contain information not supported "
        f"by the source document.\n\n"
        f"Evidence from the source:\n{evidence_text}\n\n"
        f"Original sentence:\n{original_sentence}\n\n"
        f"Rewrite the sentence using only information from the evidence. "
        f"Do not add anything not in the evidence. "
        f'If impossible, respond with exactly "UNSUPPORTED".'
    )
    result = predict_one_eg_mistral({"prompt_input": prompt})
    if not result or result.strip().upper() == "UNSUPPORTED":
        return None
    return result.strip()



def verify_summary(raw_summary, source_document):
    summary_sentences = split_sentences(raw_summary)
    source_sentences  = split_sentences(source_document)

    # edge case — empty summary or source
    if not summary_sentences or not source_sentences:
        return raw_summary, []

    verified   = []
    halluc_log = []

    for i, sentence in enumerate(summary_sentences):

        # Stage 5 — retrieve evidence
        evidence = retrieve_evidence(sentence, source_sentences)

        # Stage 6 — check entailment
        label, score = check_entailment(sentence, evidence)

        if label == "entailed":
            verified.append(sentence)
            continue

        # sentence flagged — log it
        log_entry = {
            "sentence_idx":   i,
            "original":       sentence,
            "label":          label,
            "score":          round(score, 4),
            "evidence":       evidence,
            "regenerated":    None,
            "regen_accepted": False,
        }

        # Stage 7 — attempt regeneration
        regenerated = None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            candidate = regenerate_sentence(sentence, evidence)
            if candidate is None:
                break
            new_label, new_score = check_entailment(candidate, evidence)
            if new_label == "entailed":
                regenerated = candidate
                log_entry["regen_accepted"] = True
                break

        log_entry["regenerated"] = regenerated
        halluc_log.append(log_entry)

        # Stage 8 — use regenerated if accepted, otherwise drop
        if regenerated:
            verified.append(regenerated)

    # Stage 8 — reassemble
    final_summary = " ".join(verified)
    return final_summary, halluc_log

print("Verification functions defined.")
